# Capítulo 2 - Exercício 6: Implementar `StandardScalerClone`

Neste exercício recriamos uma versão simples do `StandardScaler`, seguindo a API do Scikit-Learn. A classe precisa aprender média e desvio padrão em `fit()`, transformar dados em `transform()`, desfazer a transformação em `inverse_transform()` e lidar corretamente com nomes de atributos.

## Plano do exercício

1. Criar a classe `StandardScalerClone` como estimador do Scikit-Learn.
2. Implementar `fit()` para aprender `mean_`, `scale_` e `n_features_in_`.
3. Implementar `transform()` para padronizar os dados.
4. Implementar `inverse_transform()` para voltar à escala original.
5. Implementar `get_feature_names_out()` com suporte a `DataFrame` e nomes passados pelo usuário.
6. Testar a classe com arrays NumPy, `DataFrame` e `check_estimator`.

## Configuração

Este exercício não precisa carregar o conjunto de dados de imóveis. Vamos usar dados sintéticos pequenos, porque o foco está na implementação do transformador.

In [1]:
import sys

import numpy as np
import pandas as pd

from packaging import version
import sklearn

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

np.random.seed(42)

## Implementando o clone do `StandardScaler`

A classe herda de `BaseEstimator` e `TransformerMixin`, como um transformador típico do Scikit-Learn. O método `_validate_data()` ajuda a verificar os dados, definir `n_features_in_` e, quando a entrada é um `DataFrame`, guardar `feature_names_in_` automaticamente.

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


class StandardScalerClone(TransformerMixin, BaseEstimator):
    def __init__(self, with_mean=True):
        self.with_mean = with_mean

    def fit(self, X, y=None):
        X = self._validate_data(X, ensure_2d=True, dtype=np.float64)
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        self.scale_[self.scale_ == 0] = 1
        return self

    def transform(self, X):
        check_is_fitted(self)
        X = self._validate_data(X, reset=False, ensure_2d=True, dtype=np.float64)
        if self.with_mean:
            X = X - self.mean_
        return X / self.scale_

    def inverse_transform(self, X):
        check_is_fitted(self)
        X = self._validate_data(X, reset=False, ensure_2d=True, dtype=np.float64)
        X = X * self.scale_
        if self.with_mean:
            X = X + self.mean_
        return X

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self)
        if input_features is None:
            if hasattr(self, "feature_names_in_"):
                return self.feature_names_in_
            return np.array([f"x{i}" for i in range(self.n_features_in_)], dtype=object)

        input_features = np.asarray(input_features, dtype=object)
        if len(input_features) != self.n_features_in_:
            raise ValueError("input_features deve ter o mesmo tamanho de n_features_in_")
        if hasattr(self, "feature_names_in_") and not np.array_equal(
            input_features, self.feature_names_in_
        ):
            raise ValueError("input_features deve ser igual a feature_names_in_")
        return input_features

## Verificando compatibilidade com a API do Scikit-Learn

`check_estimator()` aplica uma bateria de testes para conferir se o objeto se comporta como um estimador válido. Se a célula não gerar erro, a estrutura básica da classe está correta.

In [3]:
from sklearn.utils.estimator_checks import check_estimator

check_estimator(StandardScalerClone())

## Testando a padronização

Agora comparamos a saída do nosso transformador com a fórmula manual: subtrair a média e dividir pelo desvio padrão de cada coluna.

In [4]:
X = np.random.rand(1000, 3)

scaler = StandardScalerClone()
X_scaled = scaler.fit_transform(X)

expected = (X - X.mean(axis=0)) / X.std(axis=0)
np.allclose(X_scaled, expected)

True

## Testando `with_mean=False`

Quando `with_mean=False`, o transformador não centraliza os dados. Ele apenas divide cada coluna pelo respectivo desvio padrão.

In [5]:
scaler = StandardScalerClone(with_mean=False)
X_scaled_uncentered = scaler.fit_transform(X)

np.allclose(X_scaled_uncentered, X / X.std(axis=0))

True

## Testando `inverse_transform()`

O método `inverse_transform()` deve desfazer a padronização. Por isso, transformar e depois inverter deve recuperar valores muito próximos dos originais.

In [6]:
scaler = StandardScalerClone()
X_back = scaler.inverse_transform(scaler.fit_transform(X))

np.allclose(X, X_back)

True

## Testando nomes genéricos de atributos

Se o transformador foi ajustado com um array NumPy, ele não conhece nomes reais de colunas. Nesse caso, `get_feature_names_out()` deve devolver nomes genéricos como `x0`, `x1` e `x2`.

In [7]:
scaler.get_feature_names_out()

array(['x0', 'x1', 'x2'], dtype=object)

In [8]:
scaler.get_feature_names_out(["a", "b", "c"])

array(['a', 'b', 'c'], dtype=object)

## Testando nomes vindos de um `DataFrame`

Quando a entrada de `fit()` é um `DataFrame`, o Scikit-Learn armazena os nomes das colunas em `feature_names_in_`. Nosso método `get_feature_names_out()` deve respeitar esses nomes.

In [9]:
df = pd.DataFrame({
    "renda_mediana": np.random.rand(100),
    "idade_mediana": np.random.rand(100),
})

scaler = StandardScalerClone()
X_scaled = scaler.fit_transform(df)

scaler.feature_names_in_, scaler.get_feature_names_out()

(array(['renda_mediana', 'idade_mediana'], dtype=object),
 array(['renda_mediana', 'idade_mediana'], dtype=object))

## Testando validações de nomes

Se o usuário passar uma lista de nomes incompatível com o `DataFrame` usado em `fit()`, o método deve gerar erro. Isso evita que um pipeline propague nomes de atributos incorretos.

In [10]:
try:
    scaler.get_feature_names_out(["a", "b"])
except ValueError as error:
    print(error)

input_features deve ser igual a feature_names_in_


## Resultado

A classe agora cobre os pontos pedidos no exercício: `fit()`, `transform()`, `inverse_transform()`, nomes de atributos e compatibilidade básica com a API do Scikit-Learn.